# Dynamic case generation (offshore-waves forcing)

Workflow:

1. Load **goals** from the case JSON → `simulations.goals_file`.
2. Open a **processed offshore-waves** zarr (`offshore_waves` in the case JSON).
3. Build the boundary forcing per timestep (`hs`, `tp`, `dir`, `spr`) for the
   numerical model (`simulations.model`: SnapWave or SWAN).
4. Add the met-ocean forcing of the case JSON (`forcing`): water level at
   the goals (e.g. CODEC) and the wind field (e.g. ERA5), if configured.
5. Run cases (optional) and postprocess to a single time-series NetCDF.

Processed offshore data must use dims `(time, goal)` with bulk variables
`hs`, `tp`, `dir`, `spr` (see `07_Download_Process_WHACS.ipynb`).

In [ ]:
from utils.config import (
    FORCING,
    SIMULATIONS,
    get_offshore_wave,
    load_goals,
    open_offshore_waves,
    open_water_level,
    open_wind,
    resolve,
    simulation_model,
)
from utils import simulation as sim

## Configuration

In [ ]:
offshore_name = "WHACS"  # must match an entry in the case JSON → offshore_waves
period = "2020"  # time slice passed to xarray .sel(time=...)

offshore_entry = get_offshore_wave(offshore_name)
model = simulation_model()
templates_dir = sim.templates_dir("dynamic")
cases_dir = sim.cases_dir("dynamic")
grid_file = sim.grid_file()
goals_file = resolve(SIMULATIONS["goals_file"])
offshore_path = resolve(offshore_entry["data_path"])
raw_goals = load_goals()

print(f"model     : {model}")
print(f"offshore  : {offshore_name} → {offshore_path}")
print(f"goals     : {goals_file}")
print(f"templates : {templates_dir}")
print(f"cases     : {cases_dir}")
print(f"grid      : {grid_file}")
print(f"period    : {period}")

## Offshore waves → boundary forcing

In [ ]:
offshore_ds = open_offshore_waves(entry=offshore_entry).sel(time=period).load()
offshore_ds

In [ ]:
# One case per time step: per-goal (= boundary node) hs/tp/dir/spr lists + tref,
# and wl from forcing.water_level when configured (0 where it has no value)
metamodel_parameters = sim.dynamic_parameters(
    offshore_ds, water_level="water_level" in FORCING
)
list(metamodel_parameters.keys())

In [ ]:
import numpy as np

# Grid maps (save_map) for some time steps. With map_reference (the dynamic
# cases folder of another run, e.g. the other model), the same hours get a map,
# so the two can be compared map by map; otherwise a random 5 %.
map_reference = None  # e.g. resolve("portal_snapwave/SnapWave_CASES_Dynamic")
n_cases = len(metamodel_parameters["tref"])
if map_reference is not None and map_reference.is_dir():
    map_trefs = {p.parent.name for p in map_reference.glob("*/output_map.*")}
    metamodel_parameters["save_map"] = [
        tref in map_trefs for tref in metamodel_parameters["tref"]
    ]
else:
    metamodel_parameters["save_map"] = np.random.choice(
        [True, False], size=n_cases, p=[0.05, 0.95]
    ).tolist()
sum(metamodel_parameters["save_map"]), n_cases

## Model wrapper

The dynamic wrapper of `simulations.model` (SnapWave or SWAN). Run
`plot_dynamic_configuration()` for a **pre-run diagnostic**: left side = domain
map + observed direction-range bars per goal; right side = text summary. Each
case is one timestep with **all goals forced** on the boundary. Review this
before `build_cases()`.

In [ ]:
# Case geometry: goals = boundary nodes, AoI enclosure, labelled output sites
setup = sim.HyWavesWrapperSetup.from_case(raw_goals)

# Wind field (forcing.wind, e.g. ERA5) at each case time; None runs without wind
wind = open_wind(period) if "wind" in FORCING else None

wrapper = setup.dynamic_wrapper(metamodel_parameters, wind=wind)
wrapper

In [ ]:
# Water level (forcing.water_level) at the goals, as given to the cases
water_level = open_water_level() if "water_level" in FORCING else None

fig = setup.plot_dynamic_configuration(
    offshore_ds,
    offshore_name=offshore_name,
    save_map=metamodel_parameters.get("save_map"),
    wind=wind,
    water_level=water_level,
)
# fig.savefig("dynamic_setup_check.png", dpi=150, bbox_inches="tight")

Met-ocean forcing of the cases: mean wind and the wind at the storm hour
(highest goal-mean offshore Hs), wind rose, wind-wave angle at the goals for
wind-sea hours (shaded: the wind correction's training range) and the water
level at the goals.

In [ ]:
fig = setup.plot_dynamic_forcing(offshore_ds, wind=wind, water_level=water_level)
# fig.savefig("dynamic_forcing_check.png", dpi=150, bbox_inches="tight")

## Build, run, postprocess

In [ ]:
wrapper.load_cases(cases_name_format=lambda ctx: f"{ctx.get('tref')}")

In [ ]:
# On the GeoOcean cluster: one SLURM array task per case. Writes case_dirs.txt
# and the array script into the cases folder (options from
# simulations.<model>.slurm), then submits it with sbatch.
# wrapper.run_cases_slurm(**sim.slurm_kwargs(max_parallel=200))
# Or locally, with the model on the PATH:
wrapper.run_cases_in_background(launcher="default", num_workers=1)

In [ ]:
wrapper.monitor_cases(value_counts="simple")

## Inspect finished cases

Post-run diagnostic of a SnapWave case (reads directly from the case folder):

- top row: full-width domain `tripcolor` map when `save_map=True` for that timestep
- bottom 2×2: `hs`, `tp`, `dir`, `spr` at output sites
- title/subtitle: case id + boundary forcing summary

Customize colormaps via `CasePlotStyle(cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"})`.

In [ ]:
from bluemath_tk.core.plotting.cases import CasePlotStyle
from utils.viz import plot_dynamic_case

style = CasePlotStyle(
    cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"}
)
plot_dynamic_case(cases_dir / "20200104 040000", style=style, show_satellite=True);

In [ ]:
result = wrapper.postprocess_cases()
out_nc = cases_dir / f"Dynamic_Cases_{offshore_name}_{period.replace('-', '')}.nc"
result.to_netcdf(out_nc)

In [ ]:
import xarray as xr

xr.open_dataset(out_nc).load()